In [1]:
import json 
from urllib.request import Request, urlopen 
 
TEAM_ID = "TEAM_56" 
API_KEY = "oc_yDTd_BxXbI8tigEoqAuHSrXrWUfcgNAa" 
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle" 
 
def api(payload): 
    req = Request(API_URL, data=json.dumps(payload).encode(), 
                  headers={"Content-Type":"application/json", "X-API-Key":API_KEY}) 
    with urlopen(req, timeout=30) as r: 
        return json.load(r) 
 
spec = api({"action":"spec", "team_id":TEAM_ID}) 
SPACE = spec["hyperparameters"] 
print("Model:", spec["model"]) 
for name, values in SPACE.items(): 
    print(name, ":", values) 

Model: ElasticNet
tol : [0.001, 0.0001]
alpha : [0.0001, 0.00020691380811147902, 0.00042813323987193956, 0.0008858667904100822, 0.0018329807108324356, 0.00379269019073225, 0.007847599703514606, 0.01623776739188721, 0.03359818286283781, 0.06951927961775606, 0.14384498882876628, 0.29763514416313164, 0.615848211066026, 1.2742749857031321, 2.6366508987303554, 5.455594781168514, 11.288378916846883, 23.357214690901213, 48.32930238571752, 100]
l1_ratio : [0.05, 0.17857142857142855, 0.3071428571428571, 0.43571428571428567, 0.5642857142857143, 0.6928571428571428, 0.8214285714285714, 0.95]
positive : [True, False]
selection : ['cyclic', 'random']
fit_intercept : [True, False]


In [2]:
def oracle_query(params):
    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })
    return float(result["loss"])


params = {
    "tol": 0.001,
    "alpha": 0.0001,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

loss = oracle_query(params)

print("Parameters:", params)
print("Loss:", loss)

Parameters: {'tol': 0.001, 'alpha': 0.0001, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.3878372069435


In [4]:
import requests

def oracle_query(params):

    data = {
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    }

    response = requests.post(
        API_URL,
        json=data,
        headers={
            "X-API-Key": API_KEY
        },
        timeout=60
    )

    result = response.json()

    return float(result["loss"])


params = {
    "tol": 0.001,
    "alpha": 0.14384498882876628,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

loss = oracle_query(params)

print("Parameters:", params)
print("Loss:", loss)

Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.3856847235907


In [6]:
import requests

def oracle_query(params):

    data = {
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    }

    response = requests.post(
        API_URL,
        json=data,
        headers={
            "X-API-Key": API_KEY
        },
        timeout=60
    )

    result = response.json()

    return float(result["loss"])

params = {
    "tol": 0.001,
    "alpha": 0.29763514416313164,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

loss = oracle_query(params)

print("Parameters:", params)
print("Loss:", loss)

Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1112100809369


In [7]:
import random

best_loss = 11.1112100809369

best_params = {
    "tol": 0.001,
    "alpha": 0.29763514416313164,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

results = []

for i in range(10):

    params = {
        "tol": random.choice(SPACE["tol"]),
        "alpha": random.choice(SPACE["alpha"]),
        "l1_ratio": random.choice(SPACE["l1_ratio"]),
        "positive": random.choice(SPACE["positive"]),
        "selection": random.choice(SPACE["selection"]),
        "fit_intercept": random.choice(SPACE["fit_intercept"])
    }

    loss = oracle_query(params)

    results.append({
        "params": params,
        "loss": loss
    })

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print()
print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.00379269019073225, 'l1_ratio': 0.95, 'positive': False, 'selection': 'cyclic', 'fit_intercept': False}
Loss: 211.40340087697
--------------------
Call: 2
Parameters: {'tol': 0.001, 'alpha': 0.03359818286283781, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': False}
Loss: 211.682847979822
--------------------
Call: 3
Parameters: {'tol': 0.0001, 'alpha': 0.0008858667904100822, 'l1_ratio': 0.6928571428571428, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.3750785146308
--------------------
Call: 4
Parameters: {'tol': 0.001, 'alpha': 0.00379269019073225, 'l1_ratio': 0.6928571428571428, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.3299213856535
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.00042813323987193956, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.3839739048038
--------------------
Call:

In [8]:
import random

for i in range(10):

    params = {
        "tol": random.choice(SPACE["tol"]),
        "alpha": random.choice(SPACE["alpha"]),
        "l1_ratio": random.choice(SPACE["l1_ratio"]),
        "positive": random.choice(SPACE["positive"]),
        "selection": "cyclic",
        "fit_intercept": True
    }

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)

Call: 1
Parameters: {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.2743395812917
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 23.357214690901213, 'l1_ratio': 0.3071428571428571, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 21.58960019162
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 11.288378916846883, 'l1_ratio': 0.5642857142857143, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 21.58960019162
--------------------
Call: 4
Parameters: {'tol': 0.001, 'alpha': 48.32930238571752, 'l1_ratio': 0.6928571428571428, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 21.58960019162
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.0001, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.387513886744
----------------

In [9]:
# Test different l1_ratio values around our best configuration

l1_values = [
    0.17857142857142855,
    0.3071428571428571,
    0.43571428571428567,
    0.5642857142857143,
    0.6928571428571428,
    0.8214285714285714,
    0.95
]

for i in range(len(l1_values)):

    params = {
        "tol": 0.001,
        "alpha": 0.29763514416313164,
        "l1_ratio": l1_values[i],
        "positive": False,
        "selection": "cyclic",
        "fit_intercept": True
    }

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.0449854934092
--------------------
Call: 2
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.0664344069035
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.43571428571428567, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1097665486939
--------------------
Call: 4
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1460579063847
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.6928571428571428, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.204

In [10]:
import random

# Keep track of configurations already tested
tested = set()

# Previous configurations
previous = [
    (0.001, 0.0001, 0.05, False, "cyclic", True),
    (0.001, 0.14384498882876628, 0.05, False, "cyclic", True),
    (0.001, 0.29763514416313164, 0.05, False, "cyclic", True),
    (0.001, 0.29763514416313164, 0.17857142857142855, False, "cyclic", True),
]

for x in previous:
    tested.add(x)

# 20 new Oracle calls
calls = 0

while calls < 20:

    params = {
        "tol": random.choice(SPACE["tol"]),
        "alpha": random.choice(SPACE["alpha"]),
        "l1_ratio": random.choice(SPACE["l1_ratio"]),
        "positive": False,
        "selection": "cyclic",
        "fit_intercept": True
    }

    key = (
        params["tol"],
        params["alpha"],
        params["l1_ratio"],
        params["positive"],
        params["selection"],
        params["fit_intercept"]
    )

    # Skip duplicate configuration
    if key in tested:
        continue

    tested.add(key)

    # Oracle call
    loss = oracle_query(params)
    calls += 1

    # Update best result
    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", calls)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print("\n==============================")
print("FINAL BEST LOSS:", best_loss)
print("FINAL BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.00042813323987193956, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 12.3825945747151
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 100, 'l1_ratio': 0.43571428571428567, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 21.58960019162
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 5.455594781168514, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 20.2018295759058
--------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.43571428571428567, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1097645167506
--------------------
Call: 5
Parameters: {'tol': 0.0001, 'alpha': 5.455594781168514, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 21.58960019162
-------

In [11]:
# 10 targeted ElasticNet searches around our current best

tests = [
    (0.14384498882876628, 0.17857142857142855),
    (0.29763514416313164, 0.17857142857142855),
    (0.615848211066026, 0.17857142857142855),

    (0.14384498882876628, 0.3071428571428571),
    (0.29763514416313164, 0.3071428571428571),
    (0.615848211066026, 0.3071428571428571),

    (0.14384498882876628, 0.05),
    (0.615848211066026, 0.05),

    (0.29763514416313164, 0.43571428571428567),
    (0.29763514416313164, 0.5642857142857143)
]

for i, (alpha, l1_ratio) in enumerate(tests):

    params = {
        "tol": 0.001,
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "positive": False,
        "selection": "cyclic",
        "fit_intercept": True
    }

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print("\n==============================")
print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.3313263265635
--------------------
Call: 2
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.0449854934092
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.615848211066026, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.4375039836054
--------------------
Call: 4
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.3180846184305
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.0664

In [12]:
# Targeted ElasticNet search
# Focus: promising alpha + l1_ratio region

tests = []

alphas = [
    0.06951927961775606,
    0.14384498882876628,
    0.29763514416313164,
    0.615848211066026,
    1.2742749857031321
]

l1_values = [
    0.05,
    0.17857142857142855,
    0.3071428571428571
]

tols = [
    0.001,
    0.0001
]

# Create combinations
for alpha in alphas:
    for l1 in l1_values:
        for tol in tols:

            params = {
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1,
                "positive": False,
                "selection": "cyclic",
                "fit_intercept": True
            }

            tests.append(params)

# Run 18 calls
for i, params in enumerate(tests):

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

print("\n==============================")
print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.7531989165228
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.7534410608639
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.72966158276
--------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.7299112659781
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.7107792085135
-----------------

In [13]:
# Next 20 targeted ElasticNet Oracle calls

tests = []

best_alpha = 0.29763514416313164

# 16 calls:
# Test positive=True for every l1_ratio and both tolerances
for l1 in SPACE["l1_ratio"]:
    for tol in SPACE["tol"]:

        tests.append({
            "tol": tol,
            "alpha": best_alpha,
            "l1_ratio": l1,
            "positive": True,
            "selection": "cyclic",
            "fit_intercept": True
        })


# 4 calls:
# Test selection="random" around the current best
random_tests = [
    0.05,
    0.17857142857142855,
    0.3071428571428571,
    0.43571428571428567
]

for l1 in random_tests:

    tests.append({
        "tol": 0.001,
        "alpha": best_alpha,
        "l1_ratio": l1,
        "positive": False,
        "selection": "random",
        "fit_intercept": True
    })


# Run exactly 20 calls
for i, params in enumerate(tests):

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")


print("\n==============================")
print("BEST LOSS:", best_loss)
print("BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.05, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1114962585133
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.05, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1114962585133
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.17857142857142855, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.097539423335
--------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.17857142857142855, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.097415574979
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.3071428571428571, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.1552740819745
--------------------
C

In [14]:
# Focused search around the current best

tests = []

# Values close to the current best
alphas = [
    0.14384498882876628,
    0.29763514416313164,
    0.615848211066026
]

l1_ratios = [
    0.05,
    0.17857142857142855,
    0.3071428571428571,
    0.43571428571428567
]

tols = [
    0.001,
    0.0001
]

# Create configurations
for alpha in alphas:
    for l1 in l1_ratios:
        for tol in tols:

            params = {
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1,
                "positive": False,
                "selection": "random",
                "fit_intercept": True
            }

            tests.append(params)


# Run Oracle calls
for i, params in enumerate(tests):

    loss = oracle_query(params)

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

    # Update best result
    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

        print("NEW BEST!")
        print("Best Loss:", best_loss)
        print("Best Parameters:", best_params)
        print("====================")


print("\n==============================")
print("FINAL BEST LOSS:", best_loss)
print("FINAL BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.05, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3857488384205
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.05, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3858310754949
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3313670570923
--------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3313887080608
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3177055440446
---------------

In [15]:
# Fine search around the current best alpha

tests = []

alphas = [
    0.06951927961775606,
    0.14384498882876628,
    0.29763514416313164,
    0.615848211066026,
    1.2742749857031321
]

l1_ratios = [
    0.05,
    0.17857142857142855,
    0.3071428571428571
]

# Try both tolerance values
for alpha in alphas:
    for l1 in l1_ratios:
        for tol in [0.001, 0.0001]:

            params = {
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1,
                "positive": False,
                "selection": "random",
                "fit_intercept": True
            }

            tests.append(params)


# Run the tests
for i, params in enumerate(tests):

    loss = oracle_query(params)

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("--------------------")

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

        print("🔥 NEW BEST!")
        print("Best Loss:", best_loss)
        print("Best Parameters:", best_params)
        print("====================")


print()
print("==============================")
print("FINAL BEST LOSS:", best_loss)
print("FINAL BEST PARAMETERS:", best_params)
print("==============================")

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.7534560565148
--------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.7535410255934
--------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.7298837387174
--------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.7299353873817
--------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.7109198877393
---------------

In [16]:
# NEXT SEARCH - NEW COMBINATIONS ONLY

tests = []

alpha = 0.29763514416313164

# Part 1:
# positive=False, selection=random
# Test the remaining l1_ratio values
l1_ratios = [
    0.5642857142857143,
    0.6928571428571428,
    0.8214285714285714,
    0.95
]

for l1 in l1_ratios:
    for tol in [0.001, 0.0001]:

        params = {
            "tol": tol,
            "alpha": alpha,
            "l1_ratio": l1,
            "positive": False,
            "selection": "random",
            "fit_intercept": True
        }

        tests.append(params)


# Part 2:
# positive=True, selection=random
# Test all l1_ratio values
l1_ratios = [
    0.05,
    0.17857142857142855,
    0.3071428571428571,
    0.43571428571428567,
    0.5642857142857143,
    0.6928571428571428,
    0.8214285714285714,
    0.95
]

for l1 in l1_ratios:
    for tol in [0.001, 0.0001]:

        params = {
            "tol": tol,
            "alpha": alpha,
            "l1_ratio": l1,
            "positive": True,
            "selection": "random",
            "fit_intercept": True
        }

        tests.append(params)


# Run the searches
for i, params in enumerate(tests):

    loss = oracle_query(params)

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("-" * 50)

    if loss < best_loss:

        best_loss = loss
        best_params = params.copy()

        print("🔥 NEW BEST!")
        print("Best Loss:", best_loss)
        print("Best Parameters:", best_params)


print("=" * 50)
print("FINAL BEST LOSS:", best_loss)
print("FINAL BEST PARAMETERS:", best_params)
print("=" * 50)

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.1454992430362
--------------------------------------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.1460450553284
--------------------------------------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.6928571428571428, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.204251793967
--------------------------------------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.6928571428571428, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.2042785945293
--------------------------------------------------
Call: 5
Parameters: {'tol': 0.001, 'alpha': 0.297635144

In [17]:
# SEARCH AROUND THE CURRENT BEST
# Only NEW / useful combinations

tests = []

alphas = [
    0.14384498882876628,
    0.29763514416313164,
    0.615848211066026
]

for alpha in alphas:

    # positive=False, selection=cyclic
    for tol in [0.001, 0.0001]:

        params = {
            "tol": tol,
            "alpha": alpha,
            "l1_ratio": 0.17857142857142855,
            "positive": False,
            "selection": "cyclic",
            "fit_intercept": True
        }

        tests.append(params)


    # positive=False, selection=random
    # only test if not already tested
    for tol in [0.001, 0.0001]:

        params = {
            "tol": tol,
            "alpha": alpha,
            "l1_ratio": 0.17857142857142855,
            "positive": False,
            "selection": "random",
            "fit_intercept": True
        }

        tests.append(params)


# Remove duplicate configurations
unique_tests = []

for params in tests:
    if params not in unique_tests:
        unique_tests.append(params)


for i, params in enumerate(unique_tests):

    loss = oracle_query(params)

    print("Call:", i + 1)
    print("Parameters:", params)
    print("Loss:", loss)
    print("-" * 60)

    if loss < best_loss:

        best_loss = loss
        best_params = params.copy()

        print("🔥 NEW BEST!")
        print("Best Loss:", best_loss)
        print("Best Parameters:", best_params)


print("=" * 60)
print("CURRENT BEST LOSS:", best_loss)
print("CURRENT BEST PARAMETERS:", best_params)
print("=" * 60)

Call: 1
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.3313263265635
------------------------------------------------------------
Call: 2
Parameters: {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
Loss: 11.3313779667129
------------------------------------------------------------
Call: 3
Parameters: {'tol': 0.001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3313670570923
------------------------------------------------------------
Call: 4
Parameters: {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
Loss: 11.3313887080608
------------------------------------------------------------
Call: 5
Pa

## CURRENT BEST LOSS: 11.0442027779871
CURRENT BEST PARAMETERS: {'tol': 0.001, 'alpha': 0.29763514416313164, 'l1_ratio': 0.17857142857142855, 'positive': False, 'selection': 'random', 'fit_intercept': True}
